In [1]:
import pandas as pd
import numpy as np

master = pd.read_csv("../data_cleaned/m3_master_with_kpis.csv")

master["date"] = pd.to_datetime(master["date"])

display(master.head())
print(master.shape)
print(master.columns)

,base_code,date,year,month_num,new_orders,shipments,unfilled_orders,total_inventories,inventories_to_shipments,unfilled_orders_to_shipments,new_orders_mom_growth,shipments_mom_growth,unfilled_orders_mom_growth,inventories_mom_growth,orders_to_shipments_ratio,inventory_to_shipments_ratio_calc
0,A11A,1992-01-01,1992,1,NaN,3459.0,NaN,3375.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.975716
1,A11A,1992-02-01,1992,2,NaN,3532.0,NaN,3345.0,NaN,NaN,NaN,0.021104,NaN,-0.008889,NaN,0.947055
2,A11A,1992-03-01,1992,3,NaN,3373.0,NaN,3345.0,NaN,NaN,NaN,-0.045017,NaN,0.000000,NaN,0.991699
3,A11A,1992-04-01,1992,4,NaN,3623.0,NaN,3291.0,NaN,NaN,NaN,0.074118,NaN,-0.016143,NaN,0.908363
4,A11A,1992-05-01,1992,5,NaN,3808.0,NaN,3349.0,NaN,NaN,NaN,0.051063,NaN,0.017624,NaN,0.879464


(165096, 16)
Index(['base_code', 'date', 'year', 'month_num', 'new_orders', 'shipments',
       'unfilled_orders', 'total_inventories', 'inventories_to_shipments',
       'unfilled_orders_to_shipments', 'new_orders_mom_growth',
       'shipments_mom_growth', 'unfilled_orders_mom_growth',
       'inventories_mom_growth', 'orders_to_shipments_ratio',
       'inventory_to_shipments_ratio_calc'],
      dtype='str')


In [2]:
risk = master.copy()

risk = risk.sort_values(["base_code", "date"])

display(risk.head())

,base_code,date,year,month_num,new_orders,shipments,unfilled_orders,total_inventories,inventories_to_shipments,unfilled_orders_to_shipments,new_orders_mom_growth,shipments_mom_growth,unfilled_orders_mom_growth,inventories_mom_growth,orders_to_shipments_ratio,inventory_to_shipments_ratio_calc
0,A11A,1992-01-01,1992,1,NaN,3459.0,NaN,3375.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.975716
1,A11A,1992-02-01,1992,2,NaN,3532.0,NaN,3345.0,NaN,NaN,NaN,0.021104,NaN,-0.008889,NaN,0.947055
2,A11A,1992-03-01,1992,3,NaN,3373.0,NaN,3345.0,NaN,NaN,NaN,-0.045017,NaN,0.000000,NaN,0.991699
3,A11A,1992-04-01,1992,4,NaN,3623.0,NaN,3291.0,NaN,NaN,NaN,0.074118,NaN,-0.016143,NaN,0.908363
4,A11A,1992-05-01,1992,5,NaN,3808.0,NaN,3349.0,NaN,NaN,NaN,0.051063,NaN,0.017624,NaN,0.879464


In [3]:
# Demand pressure: new orders growing faster than shipments
risk["demand_pressure"] = risk["new_orders_mom_growth"] - risk["shipments_mom_growth"]

# Backlog pressure: unfilled orders increasing
risk["backlog_pressure"] = risk["unfilled_orders_mom_growth"]

# Inventory pressure: inventories increasing compared to shipments
risk["inventory_pressure"] = risk["inventories_mom_growth"]

# Orders-to-shipments pressure
risk["orders_shipments_pressure"] = risk["orders_to_shipments_ratio"] - 1

In [4]:
risk["demand_pressure_flag"] = np.where(risk["demand_pressure"] > 0.03, 1, 0)
risk["backlog_pressure_flag"] = np.where(risk["backlog_pressure"] > 0.03, 1, 0)
risk["inventory_pressure_flag"] = np.where(risk["inventory_pressure"] > 0.03, 1, 0)
risk["orders_shipments_flag"] = np.where(risk["orders_shipments_pressure"] > 0.05, 1, 0)

In [5]:
risk["risk_score"] = (
    risk["demand_pressure_flag"]
    + risk["backlog_pressure_flag"]
    + risk["inventory_pressure_flag"]
    + risk["orders_shipments_flag"]
)

risk["risk_level"] = np.where(
    risk["risk_score"] >= 2,
    "High",
    np.where(risk["risk_score"] == 1, "Medium", "Low")
)

display(risk[
    [
        "base_code",
        "date",
        "new_orders",
        "shipments",
        "total_inventories",
        "unfilled_orders",
        "risk_score",
        "risk_level"
    ]
].tail(20))

,base_code,date,new_orders,shipments,total_inventories,unfilled_orders,risk_score,risk_level
165076,UTGP,2025-05-01,4660.0,4845.0,12947.0,39578.0,0,Low
165077,UTGP,2025-06-01,4600.0,4749.0,12921.0,39429.0,0,Low
165078,UTGP,2025-07-01,4541.0,4619.0,13082.0,39351.0,0,Low
165079,UTGP,2025-08-01,4615.0,4633.0,12538.0,39333.0,0,Low
165080,UTGP,2025-09-01,4586.0,4704.0,12845.0,39215.0,0,Low
165081,UTGP,2025-10-01,5149.0,5101.0,12912.0,39263.0,1,Medium
165082,UTGP,2025-11-01,4813.0,4638.0,12840.0,39438.0,0,Low
165083,UTGP,2025-12-01,4941.0,4929.0,12111.0,39450.0,0,Low
165084,UTGP,2026-01-01,4809.0,4650.0,13034.0,39609.0,1,Medium
165085,UTGP,2026-02-01,5117.0,4972.0,13426.0,39754.0,1,Medium


In [6]:
risk_dashboard = risk.dropna(
    subset=["new_orders", "shipments", "total_inventories", "unfilled_orders"]
).copy()

risk_dashboard.to_csv("../outputs/supply_chain_risk_output.csv", index=False)

print("Clean risk dashboard output saved.")

Clean risk dashboard output saved.


In [7]:
risk_dashboard["risk_level"].value_counts()

risk_level
Low       23171
Medium     8298
High       6251
Name: count, dtype: int64